<a href="https://colab.research.google.com/github/kalina-tech/exploratory-analysis-pythonn/blob/main/AB_test_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Link to dashboard https://public.tableau.com/app/profile/illia.kalinchuk/viz/ABtest_17890290636060/Dashboard1?publish=yes


In [1]:
from google.colab import auth
from google.cloud import bigquery
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.api as sm

auth.authenticate_user()
project_id = 'data-analytics-mate'
dataset_name = 'DA'

client = bigquery.Client(project=project_id)

query = f"""
with session_info as (
    SELECT
        s.date,
        s.ga_session_id,
        sp.country,
        sp.device,
        sp.continent,
        sp.channel,
        ab.test,
        ab.test_group
    FROM `{project_id}.{dataset_name}.ab_test` ab
    JOIN `{project_id}.{dataset_name}.session` s
        ON ab.ga_session_id = s.ga_session_id
    JOIN `{project_id}.{dataset_name}.session_params` sp
        ON sp.ga_session_id = ab.ga_session_id
),
session_with_orders as (
    SELECT
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group,
        count(distinct o.ga_session_id) as session_with_orders
    FROM `{project_id}.{dataset_name}.order` o
    JOIN session_info
        ON o.ga_session_id = session_info.ga_session_id
    GROUP BY
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group
),
events as (
    SELECT
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group,
        sp.event_name,
        count(sp.ga_session_id) as event_cnt
    FROM `{project_id}.{dataset_name}.event_params` sp
    JOIN session_info
        ON sp.ga_session_id = session_info.ga_session_id
    GROUP BY
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group,
        sp.event_name
),
session as (
    SELECT
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group,
        count(distinct session_info.ga_session_id) as session_cnt
    FROM session_info
    GROUP BY
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group
),
account as (
    SELECT
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group,
        count(distinct acs.ga_session_id) as new_account_cnt
    FROM `{project_id}.{dataset_name}.account_session` acs
    JOIN session_info
        ON acs.ga_session_id = session_info.ga_session_id
    GROUP BY
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group
)

SELECT
    session_with_orders.date,
    session_with_orders.country,
    session_with_orders.device,
    session_with_orders.continent,
    session_with_orders.channel,
    session_with_orders.test,
    session_with_orders.test_group,
    'session with orders' as event_name,
    session_with_orders.session_with_orders as value
FROM session_with_orders

UNION ALL

SELECT
    events.date,
    events.country,
    events.device,
    events.continent,
    events.channel,
    events.test,
    events.test_group,
    event_name,
    event_cnt as value
FROM events

UNION ALL

SELECT
    session.date,
    session.country,
    session.device,
    session.continent,
    session.channel,
    session.test,
    session.test_group,
    'session' as event_name,
    session_cnt as value
FROM session

UNION ALL

SELECT
    account.date,
    account.country,
    account.device,
    account.continent,
    account.channel,
    account.test,
    account.test_group,
    'new account' as event_name,
    new_account_cnt as value
FROM account;
"""

df = client.query(query).to_dataframe(create_bqstorage_client=False)

df.to_csv('AB_dataset.csv', index=False, encoding='utf-8')

In [2]:
df.head()

,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-11-03,New Zealand,tablet,Oceania,Organic Search,2,1,new account,1
1,2020-11-04,Tunisia,mobile,Africa,Organic Search,2,1,new account,1
2,2020-11-09,Jamaica,mobile,Americas,Organic Search,2,1,new account,1
3,2020-11-10,Puerto Rico,desktop,Americas,Paid Search,2,1,new account,1
4,2020-11-10,Iraq,mobile,Asia,Social Search,2,2,new account,1


In [3]:
print(df['event_name'].unique())


['new account' 'session with orders' 'session' 'user_engagement'
 'first_visit' 'page_view' 'scroll' 'add_shipping_info' 'view_promotion'
 'session_start' 'view_item' 'begin_checkout' 'add_to_cart'
 'view_search_results' 'add_payment_info' 'select_promotion' 'select_item'
 'click' 'view_item_list']


In [4]:
import pandas as pd
import statsmodels.api as sm


key_metrics = {
    'add_payment': 'add_payment_info',
    'add_shipping': 'add_shipping_info',
    'begin_checkout': 'begin_checkout',
    'new_accounts': 'new account'}

results_list = []


for test_name in df['test'].unique():
    df_test = df[df['test'] == test_name]


    session_cnt = df_test[df_test['event_name'] == 'session'].groupby('test_group')['value'].sum()
    total_control = session_cnt.get(1, 0)
    total_test = session_cnt.get(2, 0)

    if total_control == 0 or total_test == 0:
        continue

    for metric_label, event_name in key_metrics.items():
        metric_cnt = df_test[df_test['event_name'] == event_name].groupby('test_group')['value'].sum()
        success_control = metric_cnt.get(1, 0)
        success_test = metric_cnt.get(2, 0)


        conv_control = (success_control / total_control) if total_control > 0 else 0
        conv_test = (success_test / total_test) if total_test > 0 else 0


        metric_change = ((conv_test - conv_control) / conv_control * 100) if conv_control > 0 else 0

        successes_array = [success_test, success_control]
        total_array = [total_test, total_control]

        z_stat, p_value = sm.stats.proportions_ztest(successes_array, total_array)
        is_significant = bool(p_value < 0.05)

        results_list.append({
            'test_number': test_name,
            'metric': metric_label,
            'numerator_event': event_name,
            'denominator_event': 'session',
            'numerator_test': success_test,
            'denominator_test': total_test,
            'conversion_rate_test': conv_test,
            'numerator_control': success_control,
            'denominator_control': total_control,
            'conversion_rate_control': conv_control,
            'metric_change': metric_change,
            'z_stat': z_stat,
            'p_value': p_value,
            'significant': str(is_significant).upper()
        })


results_df = pd.DataFrame(results_list)

display(results_df)


results_df.to_csv('ab_test_results.csv', index=False, sep=';', encoding='utf-8-sig')


,test_number,metric,numerator_event,denominator_event,numerator_test,denominator_test,conversion_rate_test,numerator_control,denominator_control,conversion_rate_control,metric_change,z_stat,p_value,significant
0,2,add_payment,add_payment_info,session,2409,50244,0.047946,2344,50637,0.046290,3.576911,1.240994,0.214608,FALSE
1,2,add_shipping,add_shipping_info,session,3510,50244,0.069859,3480,50637,0.068724,1.650995,0.709557,0.477979,FALSE
2,2,begin_checkout,begin_checkout,session,4313,50244,0.085841,4262,50637,0.084168,1.988164,0.952898,0.340642,FALSE
3,2,new_accounts,new account,session,4184,50244,0.083274,4165,50637,0.082252,1.241934,0.588793,0.556000,FALSE
4,1,add_payment,add_payment_info,session,2229,45193,0.049322,1988,45362,0.043825,12.542021,3.924884,0.000087,TRUE
5,1,add_shipping,add_shipping_info,session,3221,45193,0.071272,3034,45362,0.066884,6.560481,2.603571,0.009226,TRUE
6,1,begin_checkout,begin_checkout,session,4021,45193,0.088974,3784,45362,0.083418,6.660587,2.978783,0.002894,TRUE
7,1,new_accounts,new account,session,3681,45193,0.081451,3823,45362,0.084278,-3.354299,-1.542883,0.122859,FALSE
8,4,add_payment,add_payment_info,session,3601,105141,0.034249,3731,105079,0.035507,-3.541234,-1.571106,0.116158,FALSE
9,4,add_shipping,add_shipping_info,session,4956,105141,0.047137,5128,105079,0.048801,-3.411125,-1.785795,0.074132,FALSE


Спочатку перебираємо всі тести й метрики, рахуючи загальну кількість сесій  та успішних дій  окремо для контролю і тесту.
Далі для кожної групи визначається відсоток конверсії, а також відносна зміна, тобто на скільки відсотків тестова група відрізняється від контрольної.
За допомогою функції proportions_ztest порівнюємо дві конверсії і отримаємо p_value.
Далі перевіряємо p_value чи більше за 0.05 чи менше, щоб дізнатись чи є зміна статистично значущою, і записуємо у вигляді булевого значення.
Готова таблиця з усіма розрахунками експортується у файл ab_test_results.csv для дашборду в Tableau.